## Wasserstein GAN for MNIST digits

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import ipywidgets as widgets
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
from torchvision import datasets, transforms
from IPython.display import display, clear_output

In [ ]:
# Hyperparameters
latent_dim = 100 # Size of random input to generator
lr = 0.0001
batch_size = 64
epochs = 30
lambda_gp = 10.0 # Gradient penalty to enforce lipschitz constraint
n_critic = 5 # Number of critic iterations /p generator iteration
checkpoint_path = 'wgan_checkpoint.pth'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # colab GPUs

# Data Preparation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]) # tanh scaling
])
dataset = datasets.MNIST(root='./', train=True, transform=transform, download=True)
dataloader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# Model Definitions

class Critic(nn.Module):
    def __init__(self):
        super(Critic, self).__init__()
        self.model = nn.Sequential(
            # Input (B, C, H, W)
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1)
        )

    def forward(self, img):
        return self.model(img)

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.init_layer = nn.Linear(latent_dim, 128 * 7 * 7)
        self.model = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), # upsample with ConvTranspose
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        out = self.init_layer(z)
        out = out.view(-1, 128, 7, 7)
        return self.model(out)

# Networks
critic = Critic().to(device)
generator = Generator().to(device)

# WGAN Gradient Penalty optimizers (no momentum for training stability)
optimizer_C = optim.Adam(critic.parameters(), lr=lr, betas=(0.0, 0.9))
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.0, 0.9))

# Gradient penalty function
def compute_gradient_penalty(critic, real_samples, fake_samples):
    # random weight term epsilon for interpolation between real and fake samples
    alpha = torch.rand((real_samples.size(0), 1, 1, 1)).to(device)

    # random interpolation between real and fake samples
    # because we can't test for liptchiz constraint everywhere
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = critic(interpolates)

    fake = torch.ones((real_samples.size(0), 1)).to(device)

    # gradients w.r.t. interpolates
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    # flatten gradients to calculate L2 norm
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

In [ ]:
# Load checkpoint from previous saves
if os.path.exists(checkpoint_path):
  checkpoint = torch.load(checkpoint_path, map_location=device)

  generator.load_state_dict(checkpoint['generator_state_dict'])
  critic.load_state_dict(checkpoint['critic_state_dict'])
  optimizer_G.load_state_dict(checkpoint['optimizer_G_state_dict'])
  optimizer_C.load_state_dict(checkpoint['optimizer_C_state_dict'])
  print("Resuming training from checkpoint")
else:
  print("No checkpoint found")

In [ ]:
# Training Loop

image_out = widgets.Output()
log_out = widgets.Output(layout={'border': '1px solid #ccc', 'height': '200px', 'overflow_y': 'auto'})
display(image_out, log_out)

for epoch in range(epochs):
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)

        # Critic Training
        optimizer_C.zero_grad()

        # image generation
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_imgs = generator(z)

        # critic scores
        real_validity = critic(real_imgs)
        fake_validity = critic(fake_imgs.detach())

        # gradient penalty calculation
        gp = compute_gradient_penalty(critic, real_imgs, fake_imgs.detach())

        # wasserstein Loss + gradient penalty
        loss_C = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gp

        loss_C.backward()
        optimizer_C.step()

        w_dist = torch.mean(real_validity) - torch.mean(fake_validity)
        with log_out:
          print(i, w_dist.item())

        # Generator Training, every n_critic steps
        if i % n_critic == 0:
            optimizer_G.zero_grad()

            # generate synthetic images
            gen_imgs = generator(z)

            # loss for generator
            loss_G = -torch.mean(critic(gen_imgs))

            loss_G.backward()
            optimizer_G.step()

    # display real time image generation to monitor performance
    with image_out:
      image_out.clear_output(wait=True)

      # generate an image grid
      img_grid = make_grid(fake_imgs.data[:25], nrow=5, normalize=True)

      # pytorch tensor (C, H, W) -> numpy array (H, W, C)
      img_grid_np = img_grid.cpu().numpy().transpose(1, 2, 0)

      # display the grid
      plt.figure(figsize=(6, 6))
      plt.imshow(img_grid_np)
      plt.axis('off')
      plt.title(f"Syntheic Images - Epoch {epoch+1}")
      plt.show()

    print(f"Epoch [{epoch+1}/{epochs}] | Loss C: {loss_C.item():.4f} | Loss G: {loss_G.item():.4f}")


In [ ]:
# Generate synthetic images on demand
z = torch.randn(25, latent_dim).to(device)
fake_imgs = generator(z)

img_grid = make_grid(fake_imgs.data, nrow=5, normalize=True)
img_grid_np = img_grid.cpu().numpy().transpose(1, 2, 0)

plt.figure(figsize=(6, 6))
plt.imshow(img_grid_np)
plt.axis('off')
plt.title("Generated Samples")
plt.show()

In [ ]:
# Save states

# Save just generator (optional)
torch.save(generator.state_dict(), 'wgan_generator.pth')

# Save checkpoint (for later training)
torch.save({
    'generator_state_dict': generator.state_dict(),
    'critic_state_dict': critic.state_dict(),
    'optimizer_G_state_dict': optimizer_G.state_dict(),
    'optimizer_C_state_dict': optimizer_C.state_dict(),
}, 'wgan_checkpoint.pth')